# バンディット問題勉強用ノートブック
このノートブックは、1状態のマルコフ決定過程（MDP）と形式的に等価なバンディット問題を説明するために作成されました。バンディット問題は、有限のアクション集合 $\mathcal{A}$ と、現在のアクションに対する報酬の確率分布 $p$ のタプル $(\mathcal{A}, p)$ で定義されます。
以下の import 文は必要なモジュールやパッケージを読み込むためのもので、最初は意味が分からなくても大丈夫です。必要ならば Python の本を参照してください。

In [ ]:
# rlutils パッケージのインストール（初回のみ）
!pip install -q git+https://github.com/uchibe/CPAI_study.git

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from ipywidgets import interact, fixed
%matplotlib inline

# 環境クラスとユーティリティを rlutils からインポート
from rlutils.envs import GaussianBandit
from rlutils.utils import argmax_random_tie
from rlutils.methods import uniform_random_method, run_experiment
from rlutils.plotting import compare_policies

## バンディット問題の例
5台のスロットマシンがあり、エージェントは5つの離散的な行動を選択できます。エージェントはアクションの選択に応じてスカラー報酬を受け取ります。目標は、期待報酬を最大化する行動を見つけることです。

**注意点**
- `GaussianBandit` は教育用のシンプルなクラスです（強化学習の環境構築に用いられる `gymnasium.Env` のサブクラスではありません）。


## 報酬を与える確率分布の可視化
この例題ではガウス分布で報酬を確率的に生成しています。
先頭の`seed=2026`となっている箇所の数値 2026 を学籍番号の数値と置き換えてからセルを実行してください。

レポートには`env.report_settings()`で表示される5個のガウス分布の平均と標準偏差を明記してください。

In [ ]:
env = GaussianBandit(seed=2026)  # 学籍番号の数字部分を設定してください
env.report_settings()
env.plot()

## 一様ランダム方策

より洗練されたアクション選択手法を検討する前に、シンプルな一様ランダム方策でベースラインを確認しましょう。この手法では、エージェントは完全にランダムにアクションを選択します。

In [ ]:
_ = uniform_random_method(env=env, number_of_steps=1000)

## $\varepsilon$-greedy方策
$\varepsilon$-greedyは、価値ベースの強化学習において探索（exploration）と活用（exploitation）のバランスをとるための手法です。現在の行動価値関数の推定値が $Q(a)$ であるとき、行動 $a$ を選択する確率は以下のように与えられます。
\begin{equation}
  \pi (a) =
  \begin{cases}
    \epsilon/\lvert A \rvert + (1 - \epsilon)/|\mathrm{argmax}_a \,Q| & \mathrm{if} \; \;
       a \in \mathrm{arg}\max_a Q(a), \\
    \epsilon/\lvert A \rvert & \mathrm{otherwise},
  \end{cases}
\end{equation}

### 課題
関数 `epsilon_greedy_policy` を完成させてください。今の実装は otherwise のところだけが書かれています。最大の価値を持つ行動に対して $(1 - \epsilon)/|\mathrm{argmax}_a \,Q|$ を加える部分を追加してください。

In [ ]:
def epsilon_greedy_policy(Q, epsilon=0.1):
    """Epsilon-greedy policy (returns a probability vector over actions).

    With random tie-breaking among argmax actions.
    """
    n_actions = Q.shape[0]
    policy = epsilon / n_actions * np.ones(n_actions)

    # ---- ここから課題 ----
    # Q の最大値を持つ行動の集合 best を求め、
    # policy[best] に (1 - epsilon) / len(best) を加える


    # ---- ここまで ----
    return policy


def plot_epsilon_greedy_policy(Q, epsilon=0.1):
    policy = epsilon_greedy_policy(Q, epsilon)
    action = np.arange(Q.shape[0])
    fig = plt.figure(figsize=(12, 5))
    axarr = fig.subplots(1, 2)
    axarr[0].bar(action, Q)
    axarr[0].set_xlabel('action a')
    axarr[0].set_ylabel('action value Q(a)')
    axarr[1].bar(action, policy, label=r'$\epsilon$ = %3.1f' % epsilon)
    axarr[1].set_xlabel('action a')
    axarr[1].set_ylabel(r'policy $\pi$ (a)')
    plt.legend()
    plt.show()

In [ ]:
Q = np.array([2.5, -2.5, 2.9, 1.2, 0.5])
interact(plot_epsilon_greedy_policy, Q=fixed(Q), epsilon=(0, 1, 0.05));

# 価値ベースの手法

エージェントは期待報酬を学習します。エージェントが行動 $a$ を選択し、スカラー報酬 $r$ を受け取ると、期待報酬の推定値は次のように更新されます。
\begin{equation*}
  Q(a) = Q(a) + \alpha (r - Q(a)), \quad \alpha = \frac{1}{N(a)}
\end{equation*}

### 課題
以下の関数では行動価値の更新式が
```python
    Q[action, t+1] =
```
となっています。右辺を書き換えて更新式を完成させてください。

In [ ]:
def value_based_method(env,
                       epsilon=0.1,
                       number_of_steps=1000,
                       initial_Q=0.0,
                       seed=None,
                       show_plot=True):
    rng = np.random.default_rng(seed)
    n_actions = env.n_actions
    Q = np.zeros((n_actions, number_of_steps + 1))
    N = np.zeros((n_actions, number_of_steps + 1))
    Q[:, 0] = initial_Q
    rewards = np.zeros(number_of_steps)
    actions = np.zeros(number_of_steps, dtype=int)

    for t in range(number_of_steps):
        policy = epsilon_greedy_policy(Q[:, t], epsilon=epsilon)
        action = int(rng.choice(n_actions, p=policy))
        reward = env.step(action)

        N[:, t+1] = N[:, t]
        N[action, t+1] += 1
        Q[:, t+1] = Q[:, t]
        alpha = 1.0 / N[action, t+1]

        # ---- 課題: 以下の1行を完成させてください ----
        # 右辺は Q[action, t], alpha, reward で書けます
        Q[action, t+1] =

    if show_plot:
        fig = plt.figure(figsize=(12, 5))
        axarr = fig.subplots(1, 2)
        for i in range(n_actions):
            axarr[0].plot(Q[i, :], label='$a%i$' % i)
        axarr[0].legend(); axarr[0].set_xlabel('steps')
        axarr[0].set_ylabel('action value Q(a)'); axarr[0].grid()
        for i in range(n_actions):
            axarr[1].plot(N[i, :], label='$N%i$' % i)
        axarr[1].legend(); axarr[1].set_xlabel('steps')
        axarr[1].set_ylabel('N(a)'); axarr[1].grid()
        plt.show()
        print('final Q:', Q[:, -1])
        print('average reward: %f' % rewards.mean())
    return dict(Q=Q, N=N, rewards=rewards, actions=actions,
                optimal_action=env.optimal_action)

In [ ]:
current_run_seed = np.random.randint(0, 1_000_000)
epsilon = 0.05
_ = value_based_method(env=env, epsilon=epsilon, seed=current_run_seed)

## UCB (Upper Confidence Bound) 方策
$$ \text{UCB}_t(a) = Q_t(a) + c \sqrt{\frac{\ln t}{N_t(a)}} $$

### 課題
以下の `select_ucb_action` 関数で `ucb_scores[action] = ???` の行を完成させてください。

In [ ]:
def select_ucb_action(Q_values, N_values, t, c=2.0, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    n_actions = Q_values.shape[0]
    ucb_scores = np.zeros(n_actions)

    untried = np.where(N_values == 0)[0]
    if len(untried) > 0:
        return rng.choice(untried)

    for action in range(n_actions):
        # ---- 課題: UCB スコアを計算してください ----
        ucb_scores[action] =

    return argmax_random_tie(ucb_scores, rng=rng)


def ucb_method(env, c=2.0, number_of_steps=1000,
               initial_Q=0.0, seed=None, show_plot=True):
    rng = np.random.default_rng(seed)
    n_actions = env.n_actions
    Q = np.zeros((n_actions, number_of_steps + 1))
    N = np.zeros((n_actions, number_of_steps + 1))
    Q[:, 0] = initial_Q
    rewards = np.zeros(number_of_steps)
    actions = np.zeros(number_of_steps, dtype=int)

    for t in range(number_of_steps):
        action = select_ucb_action(Q[:, t], N[:, t], t + 1, c=c, rng=rng)
        reward = env.step(action)
        N[:, t+1] = N[:, t]; N[action, t+1] += 1
        Q[:, t+1] = Q[:, t]
        alpha = 1.0 / N[action, t+1]
        Q[action, t+1] = Q[action, t] + alpha * (reward - Q[action, t])
        rewards[t] = reward; actions[t] = action

    if show_plot:
        fig = plt.figure(figsize=(12, 5))
        axarr = fig.subplots(1, 2)
        for i in range(n_actions):
            axarr[0].plot(Q[i, :], label='$a%i$' % i)
        axarr[0].legend(); axarr[0].set_xlabel('steps')
        axarr[0].set_ylabel('action value Q(a)'); axarr[0].grid()
        for i in range(n_actions):
            axarr[1].plot(N[i, :], label='$N%i$' % i)
        axarr[1].legend(); axarr[1].set_xlabel('steps')
        axarr[1].set_ylabel('N(a)'); axarr[1].grid()
        plt.show()
        print('final Q:', Q[:, -1])
        print('average reward: %f' % rewards.mean())
    return dict(Q=Q, N=N, rewards=rewards, actions=actions,
                optimal_action=env.optimal_action)

In [ ]:
current_run_seed = np.random.randint(0, 1_000_000)
_ = ucb_method(env=env, c=2.0, seed=current_run_seed)

## 複数回の実行による平均化と比較

探索戦略を公平に評価するために、多くの独立したシードで平均をとります。

In [ ]:
n_runs = 200
T = 1000

configs = [
    dict(label=r'$\varepsilon$=0.1, $Q_0$=0',  method_function=value_based_method, epsilon=0.1,  initial_Q=0.0),
    dict(label=r'$\varepsilon$=0.01, $Q_0$=0', method_function=value_based_method, epsilon=0.01, initial_Q=0.0),
    dict(label=r'greedy, $Q_0$=10 (optimistic)', method_function=value_based_method, epsilon=0.0,  initial_Q=10.0),
    dict(label=r'UCB c=2', method_function=ucb_method, c=2.0, initial_Q=0.0),
]

results = []
for cfg in configs:
    current_cfg = cfg.copy()
    label = current_cfg.pop('label')
    method_func = current_cfg.pop('method_function')
    avg_r, opt_r, Q_last = run_experiment(env=env, method_function=method_func,
                                          n_runs=n_runs, number_of_steps=T, **current_cfg)
    results.append((label, avg_r, opt_r, Q_last))

fig, axarr = plt.subplots(1, 2, figsize=(13, 5))
for label, avg_r, opt_r, _ in results:
    axarr[0].plot(avg_r, label=label)
    axarr[1].plot(opt_r, label=label)
axarr[0].set_xlabel('steps'); axarr[0].set_ylabel('average reward')
axarr[0].grid(); axarr[0].legend()
axarr[1].set_xlabel('steps'); axarr[1].set_ylabel('optimal action rate')
axarr[1].grid(); axarr[1].legend()
plt.tight_layout(); plt.show()